In [ ]:
!nvidia-smi
!pip install -q transformers accelerate pillow datasets pandas tqdm pyarrow

In [2]:
import os
import gc
import re
import json
import shutil
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter
from datasets import load_dataset
from tqdm.auto import tqdm
from PIL import Image
from transformers import LlavaOnevisionForConditionalGeneration, AutoProcessor

In [3]:
# ============================================================
# LLaVA-OneVision Original Split — Sequence-Level Pipeline + PMI
# Original image first, resized fallback if needed
# ============================================================# ============================================================
# 1. Configuration
# ============================================================

MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
DATASET_NAME = "anvo25/vlms-are-biased"
SPLIT_NAME = "original"

OUTPUT_DIR = Path("/kaggle/working/llava_original_sequence_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 2
MIN_CANDIDATE = 0
EXCLUDE_OPTICAL_ILLUSIONS = True

# Use 20 for smoke test; None for full run.
MAX_SAMPLES = None

# Sequence pipeline is numeric-only.
RUN_ID_ROWS = False

MAX_NEW_NUMERIC = 3
GENERATION_NUMERIC_SUFFIX = "\nAnswer with exactly one integer. No words."

# Original no-resize safety.
# If original LLaVA processing creates more than this many crops,
# skip original generation and retry resized immediately.
MAX_CROPS_ORIGINAL = 5

# Resized fallback.
MAX_IMAGE_SIDE_RETRY = 448

SAVE_EVERY = 10

print("MODEL:", MODEL_ID)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MAX_CROPS_ORIGINAL:", MAX_CROPS_ORIGINAL)
print("MAX_IMAGE_SIDE_RETRY:", MAX_IMAGE_SIDE_RETRY)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))



MODEL: llava-hf/llava-onevision-qwen2-7b-ov-hf
OUTPUT_DIR: /kaggle/working/llava_original_sequence_outputs
MAX_CROPS_ORIGINAL: 5
MAX_IMAGE_SIDE_RETRY: 448
CUDA: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [4]:
# ============================================================
# 2. Cleanup
# ============================================================

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            try:
                with torch.cuda.device(i):
                    torch.cuda.empty_cache()
            except Exception:
                pass
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


cleanup()


# ============================================================
# 3. Load model
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

max_memory = None
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    max_memory = {i: "14GiB" for i in range(torch.cuda.device_count())}
    max_memory["cpu"] = "30GiB"

print("Loading model...")
print("max_memory:", max_memory)

model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
tokenizer.padding_side = "left"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id
model.eval()

print("Loaded model.")
cleanup()


Loading model...
max_memory: {0: '14GiB', 1: '14GiB', 'cpu': '30GiB'}


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/765 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/621 [00:00<?, ?B/s]

Loaded model.


In [5]:
# ============================================================
# 4. Load dataset
# ============================================================

ds = load_dataset(DATASET_NAME, split=SPLIT_NAME)
print("Original rows:", len(ds))

if EXCLUDE_OPTICAL_ILLUSIONS:
    before = len(ds)
    ds = ds.filter(
        lambda x: str(x.get("topic", "")).strip().lower() != "optical illusion"
    )
    print("Removed Optical Illusion rows:", before - len(ds))

if MAX_SAMPLES is not None:
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))
    print("Debug rows:", len(ds))

if "idx" in ds.column_names:
    ds = ds.remove_columns(["idx"])

ds = ds.add_column("idx", list(range(len(ds))))

assert ds["idx"][0] == 0
assert ds["idx"][-1] == len(ds) - 1
assert len(set(ds["idx"])) == len(ds)

print("Final rows:", len(ds))
print("Topics:", Counter(ds["topic"]))


README.md: 0.00B [00:00, ?B/s]

data/main-00000-of-00002.parquet:   0%|          | 0.00/25.0M [00:00<?, ?B/s]

data/main-00001-of-00002.parquet:   0%|          | 0.00/375M [00:00<?, ?B/s]

data/identification-00000-of-00001.parqu(…):   0%|          | 0.00/391M [00:00<?, ?B/s]

data/withtitle-00000-of-00001.parquet:   0%|          | 0.00/451M [00:00<?, ?B/s]

...

Original rows: 458


Filter:   0%|          | 0/458 [00:00<?, ? examples/s]

Removed Optical Illusion rows: 132


Flattening the indices:   0%|          | 0/326 [00:00<?, ? examples/s]

Final rows: 326
Topics: Counter({'Animals': 182, 'Logos': 94, 'Flags': 38, 'Game Boards': 8, 'Chess Pieces': 4})


In [12]:
# ============================================================
# 5. Helpers
# ============================================================

BRACE_RE = re.compile(r"\{([^{}]+)\}")
INT_RE = re.compile(r"-?\d+")


def text(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()


def unbrace(x):
    x = text(x)
    m = BRACE_RE.search(x)
    return m.group(1).strip() if m else x


def first_int(x):
    m = INT_RE.search(unbrace(x))
    return int(m.group(0)) if m else None


def norm(x):
    return unbrace(x).lower().strip().rstrip(".")


def is_numeric(row):
    qtype = text(row.get("type_of_question", "")).lower()
    return ("count" in qtype or "illusion" in qtype) and first_int(row.get("ground_truth")) is not None


def candidate_window(gt):
    gt = int(gt)
    return list(range(max(MIN_CANDIDATE, gt - WINDOW), gt + WINDOW + 1))


def resize_image(image, max_side=MAX_IMAGE_SIDE_RETRY):
    image = image.convert("RGB")
    original_size = image.size

    w, h = image.size
    scale = min(1.0, max_side / max(w, h))

    if scale < 1.0:
        image = image.resize(
            (int(w * scale), int(h * scale)),
            Image.Resampling.LANCZOS,
        )

    return image, original_size, image.size


def chat_prompt(prompt):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt},
        ],
    }]
    return processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def entropy_from_logprobs(log_probs):
    probs = log_probs.exp()
    return float(-(probs * log_probs).sum().item())


def is_retryable_error(err):
    err = text(err).lower()
    return (
        "cuda" in err
        or "oom" in err
        or "out of memory" in err
        or "launch failure" in err
        or "too_many_image_crops" in err
    )


def sort_key(row):
    return int(row.get("idx", 10**12))


def read_records(path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        try:
            return pd.read_csv(path).to_dict("records")
        except pd.errors.EmptyDataError:
            return []
    return []


def save_parquet(rows, path):
    df = pd.DataFrame(rows)
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype(str)
    df.to_parquet(path, index=False)


def safe_num_crops(inputs):
    if "pixel_values" not in inputs:
        return 1, None
    shape = tuple(inputs["pixel_values"].shape)
    crops = shape[1] if len(shape) == 5 else 1
    return crops, shape

In [13]:


# ============================================================
# 6. Output state
# ============================================================

MODEL_OUTPUTS_CSV = OUTPUT_DIR / "llava_original_sequence_model_outputs.csv"
CANDIDATES_CSV = OUTPUT_DIR / "llava_original_sequence_candidates.csv"
RESULTS_CSV = OUTPUT_DIR / "llava_original_sequence_results.csv"
SKIPPED_CSV = OUTPUT_DIR / "llava_original_sequence_skipped.csv"
RESULTS_PMI_CSV = OUTPUT_DIR / "llava_original_sequence_results_with_pmi.csv"

model_rows = read_records(MODEL_OUTPUTS_CSV)
candidate_rows = read_records(CANDIDATES_CSV)
result_rows = read_records(RESULTS_CSV)
skipped_rows = read_records(SKIPPED_CSV)

print("Loaded checkpoint:")
print("model_rows:", len(model_rows))
print("candidate_rows:", len(candidate_rows))
print("result_rows:", len(result_rows))
print("skipped_rows:", len(skipped_rows))


def save_outputs():
    pd.DataFrame(sorted(model_rows, key=sort_key)).to_csv(
        MODEL_OUTPUTS_CSV,
        index=False,
    )

    pd.DataFrame(
        sorted(candidate_rows, key=lambda r: (sort_key(r), int(r.get("candidate", -1))))
    ).to_csv(
        CANDIDATES_CSV,
        index=False,
    )

    pd.DataFrame(sorted(result_rows, key=sort_key)).to_csv(
        RESULTS_CSV,
        index=False,
    )

    pd.DataFrame(sorted(skipped_rows, key=sort_key)).to_csv(
        SKIPPED_CSV,
        index=False,
    )

Loaded checkpoint:
model_rows: 0
candidate_rows: 0
result_rows: 0
skipped_rows: 0


In [14]:
# ============================================================
# 7. Base row
# ============================================================

def make_base_row(
    row,
    prompt_used,
    answer,
    error,
    original_size,
    processed_size,
    input_shape=None,
    pixel_shape=None,
    num_crops=None,
    run_mode="original",
    fallback_used=False,
    original_error_before_fallback="",
):
    gt_raw = row.get("ground_truth", "")
    gt_num = first_int(gt_raw)
    pred_num = first_int(answer)

    return {
        "idx": int(row["idx"]),
        "ID": row.get("ID"),
        "image_path": row.get("image_path"),
        "topic": row.get("topic"),
        "sub_topic": row.get("sub_topic"),
        "question_type": text(row.get("type_of_question", "")),
        "type_of_question": text(row.get("type_of_question", "")),
        "prompt": text(row.get("prompt", "")),
        "prompt_used_for_generation": prompt_used,
        "ground_truth": str(gt_num) if gt_num is not None else text(gt_raw),
        "ground_truth_raw": gt_raw,
        "ground_truth_num": gt_num,
        "model_answer": answer,
        "pred_norm": norm(answer),
        "gt_norm": norm(gt_raw),
        "pred_num": pred_num,
        "correct_exact": norm(answer) == norm(gt_raw),
        "correct_numeric": (
            int(pred_num) == int(gt_num)
            if pred_num is not None and gt_num is not None
            else False
        ),
        "generation_error": error,

        "run_mode": run_mode,
        "fallback_used": bool(fallback_used),
        "original_error_before_fallback": original_error_before_fallback,
        "image_resize_applied": run_mode == "resized_fallback",
        "max_image_side": MAX_IMAGE_SIDE_RETRY if run_mode == "resized_fallback" else None,

        "original_image_width": int(original_size[0]),
        "original_image_height": int(original_size[1]),
        "processed_image_width": int(processed_size[0]),
        "processed_image_height": int(processed_size[1]),

        "input_ids_shape": str(input_shape) if input_shape is not None else "",
        "pixel_values_shape": str(pixel_shape) if pixel_shape is not None else "",
        "num_image_crops": num_crops,
    }


# ============================================================
# 8. Generation attempt + fallback
# ============================================================

def prepare_image_for_mode(row, run_mode):
    if run_mode == "original":
        image = row["image"].convert("RGB")
        original_size = image.size
        processed_size = image.size
        return image, original_size, processed_size

    if run_mode == "resized_fallback":
        return resize_image(row["image"], MAX_IMAGE_SIDE_RETRY)

    raise ValueError(f"Unknown run_mode: {run_mode}")


def generate_attempt(row, max_new_tokens, need_scores, run_mode):
    row = dict(row)

    prompt_raw = text(row.get("prompt", ""))
    prompt_used = prompt_raw + GENERATION_NUMERIC_SUFFIX

    original_size = (-1, -1)
    processed_size = (-1, -1)
    input_shape = None
    pixel_shape = None
    num_crops = None

    inputs = None
    output = None
    gen_logprobs = None

    try:
        image, original_size, processed_size = prepare_image_for_mode(row, run_mode)

        inputs = processor(
            images=image,
            text=chat_prompt(prompt_used),
            return_tensors="pt",
        ).to(device)

        input_shape = tuple(inputs["input_ids"].shape)
        num_crops, pixel_shape = safe_num_crops(inputs)

        print(
            "idx:", row["idx"],
            "mode:", run_mode,
            "orig:", original_size,
            "processed:", processed_size,
            "input_ids:", input_shape,
            "pixel_values:", pixel_shape,
            "crops:", num_crops,
        )

        if run_mode == "original" and num_crops > MAX_CROPS_ORIGINAL:
            return {
                "row": row,
                "prompt_used": prompt_used,
                "answer": "",
                "gen_ids": None,
                "gen_logprobs": None,
                "error": f"too_many_image_crops_original:{num_crops}",
                "image_for_scoring": None,
                "original_size": original_size,
                "processed_size": processed_size,
                "input_shape": input_shape,
                "pixel_shape": pixel_shape,
                "num_crops": num_crops,
                "run_mode": run_mode,
            }

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                return_dict_in_generate=True,
                output_scores=need_scores,
                pad_token_id=tokenizer.pad_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]
        gen_ids = output.sequences[0, prompt_len:].detach().cpu()
        answer = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        if need_scores and len(output.scores) > 0:
            gen_logprobs = torch.log_softmax(
                torch.stack(output.scores, dim=0).squeeze(1).float(),
                dim=-1,
            ).detach().cpu()

        return {
            "row": row,
            "prompt_used": prompt_used,
            "answer": answer,
            "gen_ids": gen_ids,
            "gen_logprobs": gen_logprobs,
            "error": "",
            "image_for_scoring": image,
            "original_size": original_size,
            "processed_size": processed_size,
            "input_shape": input_shape,
            "pixel_shape": pixel_shape,
            "num_crops": num_crops,
            "run_mode": run_mode,
        }

    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM at idx:", row.get("idx"), "mode:", run_mode)

        return {
            "row": row,
            "prompt_used": prompt_used,
            "answer": "",
            "gen_ids": None,
            "gen_logprobs": None,
            "error": f"cuda_oom_generation_{run_mode}",
            "image_for_scoring": None,
            "original_size": original_size,
            "processed_size": processed_size,
            "input_shape": input_shape,
            "pixel_shape": pixel_shape,
            "num_crops": num_crops,
            "run_mode": run_mode,
        }

    except RuntimeError as e:
        err = repr(e)

        if "unspecified launch failure" in err.lower() or "cudaerrorlaunchfailure" in err.lower():
            err = f"cuda_launch_failure_generation_{run_mode}: " + err
            print("CUDA launch failure at idx:", row.get("idx"))
            print("Saving and stopping. Restart accelerator before continuing.")
            save_outputs()
            raise RuntimeError(err)

        if "cuda" in err.lower() or "oom" in err.lower() or "out of memory" in err.lower():
            err = f"cuda_runtime_generation_{run_mode}: " + err

        print("Runtime error at idx:", row.get("idx"), "mode:", run_mode, err)

        return {
            "row": row,
            "prompt_used": prompt_used,
            "answer": "",
            "gen_ids": None,
            "gen_logprobs": None,
            "error": err,
            "image_for_scoring": None,
            "original_size": original_size,
            "processed_size": processed_size,
            "input_shape": input_shape,
            "pixel_shape": pixel_shape,
            "num_crops": num_crops,
            "run_mode": run_mode,
        }

    except Exception as e:
        print("Error at idx:", row.get("idx"), "mode:", run_mode, repr(e))

        return {
            "row": row,
            "prompt_used": prompt_used,
            "answer": "",
            "gen_ids": None,
            "gen_logprobs": None,
            "error": repr(e),
            "image_for_scoring": None,
            "original_size": original_size,
            "processed_size": processed_size,
            "input_shape": input_shape,
            "pixel_shape": pixel_shape,
            "num_crops": num_crops,
            "run_mode": run_mode,
        }

    finally:
        try:
            del inputs, output, gen_logprobs
        except Exception:
            pass
        cleanup()


def generate_with_fallback(row, max_new_tokens, need_scores):
    original = generate_attempt(
        row=row,
        max_new_tokens=max_new_tokens,
        need_scores=need_scores,
        run_mode="original",
    )

    if not is_retryable_error(original["error"]):
        original["fallback_used"] = False
        original["original_error_before_fallback"] = ""
        return original

    fallback = generate_attempt(
        row=row,
        max_new_tokens=max_new_tokens,
        need_scores=need_scores,
        run_mode="resized_fallback",
    )

    fallback["fallback_used"] = True
    fallback["original_error_before_fallback"] = original["error"]

    return fallback

In [15]:
# ============================================================
# 9. Memory-safe sequence candidate scoring
# ============================================================

def score_candidate_sequence(image, prompt, answer):
    """
    Teacher-forced sequence scoring for one candidate.

    Uses raw dataset prompt, not generation suffix.
    Scores only the answer-token span.
    Converts only one vocab vector to float32 at a time.
    """
    prompt_text = chat_prompt(prompt)
    answer_text = str(answer)
    answer_ids_expected = tokenizer.encode(answer_text, add_special_tokens=False)
    full_text = prompt_text + answer_text

    prompt_inputs = None
    inputs = None
    output = None

    try:
        prompt_inputs = processor(
            images=image,
            text=prompt_text,
            return_tensors="pt",
        )
        prompt_len = prompt_inputs["input_ids"].shape[1]

        inputs = processor(
            images=image,
            text=full_text,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            output = model(**inputs)

        input_ids = inputs["input_ids"]
        ids = input_ids[0].detach().cpu().tolist()

        span_start = prompt_len
        span_end = prompt_len + len(answer_ids_expected)
        span_mode = "exact"

        if ids[span_start:span_end] != answer_ids_expected:
            span_start = None
            search_from = max(0, prompt_len - 5)

            for pos in range(search_from, len(ids) - len(answer_ids_expected) + 1):
                if ids[pos:pos + len(answer_ids_expected)] == answer_ids_expected:
                    span_start = pos
                    span_end = pos + len(answer_ids_expected)
                    span_mode = "fallback"
                    break

            if span_start is None:
                return {
                    "status": "failed",
                    "span_mode": "failed",
                    "error": "candidate_span_not_found",
                    "candidate_token_ids": json.dumps(answer_ids_expected),
                }

        token_lps = []
        token_entropies = []

        for pos in range(span_start, span_end):
            if pos == 0:
                return {
                    "status": "failed",
                    "span_mode": "failed",
                    "error": "span_starts_at_zero",
                    "candidate_token_ids": json.dumps(answer_ids_expected),
                }

            tok_id = int(input_ids[0, pos].item())

            pred_logits = output.logits[0, pos - 1, :].float()
            pred_log_probs = torch.log_softmax(pred_logits, dim=-1)

            token_lps.append(float(pred_log_probs[tok_id].item()))
            token_entropies.append(entropy_from_logprobs(pred_log_probs))

            del pred_logits, pred_log_probs

        seq_lp = float(sum(token_lps))
        n_tokens = len(token_lps)

        return {
            "status": "ok",
            "span_mode": span_mode,
            "seq_logprob": seq_lp,
            "avg_logprob": seq_lp / max(1, n_tokens),
            "char_logprob": seq_lp / max(1, len(answer_text)),
            "entropy_step1": token_entropies[0] if token_entropies else None,
            "entropy_mean": float(np.mean(token_entropies)) if token_entropies else None,
            "n_tokens": n_tokens,
            "candidate_token_ids": json.dumps(answer_ids_expected),
            "token_logprobs": json.dumps(token_lps),
        }

    except torch.cuda.OutOfMemoryError:
        return {
            "status": "failed",
            "span_mode": "failed",
            "error": "cuda_oom_sequence_scoring",
            "candidate_token_ids": json.dumps(answer_ids_expected),
        }

    except RuntimeError as e:
        err = repr(e)

        if "unspecified launch failure" in err.lower() or "cudaerrorlaunchfailure" in err.lower():
            print("CUDA launch failure during sequence scoring.")
            save_outputs()
            raise RuntimeError("cuda_launch_failure_sequence_scoring: " + err)

        if "cuda" in err.lower() or "oom" in err.lower() or "out of memory" in err.lower():
            err = "cuda_runtime_sequence_scoring: " + err

        return {
            "status": "failed",
            "span_mode": "failed",
            "error": err,
            "candidate_token_ids": json.dumps(answer_ids_expected),
        }

    except Exception as e:
        return {
            "status": "failed",
            "span_mode": "failed",
            "error": repr(e),
            "candidate_token_ids": json.dumps(answer_ids_expected),
        }

    finally:
        try:
            del prompt_inputs, inputs, output
        except Exception:
            pass
        cleanup()


def score_candidate_window(image, prompt, candidates):
    scored = []

    for candidate in candidates:
        score = score_candidate_sequence(
            image=image,
            prompt=prompt,
            answer=str(candidate),
        )

        scored.append({
            "candidate": int(candidate),
            "candidate_str": str(candidate),
            **score,
        })

    return scored
    

In [16]:


# ============================================================
# 10. Numeric sequence analysis
# ============================================================

def analyze_numeric_sequence(result):
    row = result["row"]
    gt = first_int(row.get("ground_truth", ""))

    if gt is None:
        return None, [], {"skip_reason": "non_numeric_ground_truth"}

    if result["error"]:
        return None, [], {"skip_reason": "generation_error"}

    image = result.get("image_for_scoring", None)
    if image is None:
        return None, [], {"skip_reason": "missing_scoring_image"}

    scoring_prompt = text(row.get("prompt", ""))

    candidates = candidate_window(gt)
    scored = score_candidate_window(image, scoring_prompt, candidates)
    ok = [x for x in scored if x.get("status") == "ok"]

    if not ok:
        return None, scored, {
            "skip_reason": "all_candidate_spans_failed",
            "candidate_set": json.dumps(candidates),
        }

    gold_rows = [x for x in ok if int(x["candidate"]) == int(gt)]
    non_gold = [x for x in ok if int(x["candidate"]) != int(gt)]

    if not gold_rows:
        return None, scored, {
            "skip_reason": "gold_span_failed",
            "candidate_set": json.dumps(candidates),
        }

    if not non_gold:
        return None, scored, {
            "skip_reason": "no_valid_non_gold_candidate",
            "candidate_set": json.dumps(candidates),
        }

    gold = gold_rows[0]
    proxy = max(non_gold, key=lambda x: float(x["avg_logprob"]))

    ranked = sorted(ok, key=lambda x: float(x["avg_logprob"]), reverse=True)
    gt_rank = next(
        i + 1 for i, x in enumerate(ranked)
        if int(x["candidate"]) == int(gt)
    )

    answer_step = None
    detected_digit = None
    detected_token_text = None
    entropy_step1 = None
    entropy_mean = None

    if result["gen_ids"] is not None:
        for i, token_id in enumerate(result["gen_ids"]):
            tok = tokenizer.decode([int(token_id)], skip_special_tokens=True)
            detected = first_int(tok)
            if detected is not None:
                answer_step = i
                detected_digit = detected
                detected_token_text = tok
                break

    if (
        answer_step is not None
        and result["gen_logprobs"] is not None
        and answer_step < result["gen_logprobs"].shape[0]
    ):
        entropy_step1 = entropy_from_logprobs(result["gen_logprobs"][answer_step].float())
        entropy_mean = float(np.mean([
            entropy_from_logprobs(result["gen_logprobs"][t].float())
            for t in range(answer_step + 1)
        ]))

    base = make_base_row(
        row=row,
        prompt_used=result["prompt_used"],
        answer=result["answer"],
        error=result["error"],
        original_size=result["original_size"],
        processed_size=result["processed_size"],
        input_shape=result["input_shape"],
        pixel_shape=result["pixel_shape"],
        num_crops=result["num_crops"],
        run_mode=result["run_mode"],
        fallback_used=result.get("fallback_used", False),
        original_error_before_fallback=result.get("original_error_before_fallback", ""),
    )

    seq_margin = float(gold["seq_logprob"] - proxy["seq_logprob"])
    avg_margin = float(gold["avg_logprob"] - proxy["avg_logprob"])
    char_margin = float(gold["char_logprob"] - proxy["char_logprob"])

    candidate_scores = torch.tensor([x["avg_logprob"] for x in ok], dtype=torch.float32)
    candidate_probs = torch.softmax(candidate_scores, dim=0)
    candidate_log_probs = torch.log_softmax(candidate_scores, dim=0)
    entropy_candidates = float(-(candidate_probs * candidate_log_probs).sum().item())

    same_length = int(gold["n_tokens"]) == int(proxy["n_tokens"])

    result_row = {
        **base,

        "expected_bias": str(proxy["candidate"]),
        "proxy_bias_value": int(proxy["candidate"]),
        "proxy_bias_definition": f"best non-GT candidate from GT±{WINDOW} by avg_logprob",

        "candidate_policy": f"GT±{WINDOW}",
        "candidate_set": json.dumps(candidates),
        "num_candidates_attempted": len(candidates),
        "num_candidates_scored": len(ok),

        "answer_step": answer_step,
        "detected_digit": detected_digit,
        "detected_token_text": detected_token_text,

        "gold_seq_logprob": gold["seq_logprob"],
        "gold_avg_logprob": gold["avg_logprob"],
        "gold_char_logprob": gold["char_logprob"],
        "gold_entropy_step1": gold["entropy_step1"],
        "gold_entropy_mean": gold["entropy_mean"],
        "gold_n_tokens": gold["n_tokens"],
        "gold_span_mode": gold["span_mode"],

        "bias_seq_logprob": proxy["seq_logprob"],
        "bias_avg_logprob": proxy["avg_logprob"],
        "bias_char_logprob": proxy["char_logprob"],
        "bias_entropy_step1": proxy["entropy_step1"],
        "bias_entropy_mean": proxy["entropy_mean"],
        "bias_n_tokens": proxy["n_tokens"],
        "bias_span_mode": proxy["span_mode"],

        "seq_margin": seq_margin,
        "avg_margin": avg_margin,
        "char_margin": char_margin,

        "gt_rank_seq": gt_rank,
        "prefers_gold": avg_margin > 0,
        "prefers_gt_seq": avg_margin > 0,
        "entropy_candidates": entropy_candidates,

        "entropy_step1": entropy_step1,
        "entropy_mean": entropy_mean,

        "generated_matches_proxy": (
            detected_digit is not None
            and int(detected_digit) == int(proxy["candidate"])
        ),
        "full_hallucination": (
            detected_digit is not None
            and int(detected_digit) != int(gt)
            and int(detected_digit) != int(proxy["candidate"])
        ),

        "same_length": same_length,
        "length_matched": same_length,
        "relative_change": abs(float(gt) - float(proxy["candidate"])) / max(1.0, abs(float(proxy["candidate"]))),
        "gt_minus_bias": float(gt) - float(proxy["candidate"]),
        "direction": (
            "GT>proxy" if float(gt) > float(proxy["candidate"])
            else "GT<proxy" if float(gt) < float(proxy["candidate"])
            else "GT=proxy"
        ),
    }

    return result_row, scored, None

In [17]:


# ============================================================
# 11. PMI
# ============================================================

def score_unconditional(answer):
    prompt = "Answer:"
    answer = str(answer)
    full_text = prompt + answer

    prompt_ids = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(device)

    full_ids = tokenizer(
        full_text,
        return_tensors="pt",
        add_special_tokens=False,
    )["input_ids"].to(device)

    prompt_len = prompt_ids.shape[1]
    output = None

    try:
        with torch.inference_mode():
            output = model(input_ids=full_ids)

        answer_ids = full_ids[0, prompt_len:]
        token_lps = []

        for i, token_id in enumerate(answer_ids):
            pos = prompt_len - 1 + i
            pred_logits = output.logits[0, pos, :].float()
            pred_log_probs = torch.log_softmax(pred_logits, dim=-1)
            token_lps.append(float(pred_log_probs[token_id].item()))
            del pred_logits, pred_log_probs

        seq_lp = float(sum(token_lps))
        n_tokens = len(token_lps)

        return {
            "seq_logprob": seq_lp,
            "avg_logprob": seq_lp / max(1, n_tokens),
            "char_logprob": seq_lp / max(1, len(answer)),
            "n_tokens": n_tokens,
        }

    finally:
        try:
            del prompt_ids, full_ids, output
        except Exception:
            pass
        cleanup()


def add_pmi_columns(rows):
    df = pd.DataFrame(rows)

    if len(df) == 0:
        return rows

    unique_values = sorted(
        set(df["ground_truth"].astype(str))
        | set(df["expected_bias"].astype(str))
    )

    print(f"Running unconditional PMI scoring for {len(unique_values)} unique values:")
    print(unique_values)

    uncond = {}

    for value in tqdm(unique_values, desc="PMI"):
        uncond[str(value)] = score_unconditional(str(value))

    df["gold_uncond_avg_lp"] = df["ground_truth"].astype(str).map(
        {k: v["avg_logprob"] for k, v in uncond.items()}
    )
    df["bias_uncond_avg_lp"] = df["expected_bias"].astype(str).map(
        {k: v["avg_logprob"] for k, v in uncond.items()}
    )

    df["gold_uncond_char_lp"] = df["ground_truth"].astype(str).map(
        {k: v["char_logprob"] for k, v in uncond.items()}
    )
    df["bias_uncond_char_lp"] = df["expected_bias"].astype(str).map(
        {k: v["char_logprob"] for k, v in uncond.items()}
    )

    df["gold_pmi"] = df["gold_avg_logprob"] - df["gold_uncond_avg_lp"]
    df["bias_pmi"] = df["bias_avg_logprob"] - df["bias_uncond_avg_lp"]
    df["pmi_margin"] = df["gold_pmi"] - df["bias_pmi"]
    df["prefers_gold_pmi"] = df["pmi_margin"] > 0

    df["gold_char_pmi"] = df["gold_char_logprob"] - df["gold_uncond_char_lp"]
    df["bias_char_pmi"] = df["bias_char_logprob"] - df["bias_uncond_char_lp"]
    df["char_pmi_margin"] = df["gold_char_pmi"] - df["bias_char_pmi"]
    df["prefers_gold_char_pmi"] = df["char_pmi_margin"] > 0

    return df.to_dict("records")


# ============================================================
# 12. Prepare numeric rows
# ============================================================

all_numeric_rows = []

for row in ds:
    row = dict(row)
    if is_numeric(row):
        all_numeric_rows.append(row)

done_sequence_idx = {
    int(r["idx"]) for r in result_rows
    if pd.notna(r.get("idx"))
}

done_sequence_idx |= {
    int(r["idx"]) for r in skipped_rows
    if r.get("pipeline") == "sequence" and pd.notna(r.get("idx"))
}

numeric_rows = [
    r for r in all_numeric_rows
    if int(r["idx"]) not in done_sequence_idx
]

print("Total numeric rows:", len(all_numeric_rows))
print("Pending numeric rows:", len(numeric_rows))

Total numeric rows: 163
Pending numeric rows: 163


In [18]:


# ============================================================
# 13. Run sequence pipeline
# ============================================================

processed = len(done_sequence_idx)
next_save = ((processed // SAVE_EVERY) + 1) * SAVE_EVERY

for row in tqdm(numeric_rows, desc="LLaVA sequence numeric rows"):
    result = generate_with_fallback(
        row=row,
        max_new_tokens=MAX_NEW_NUMERIC,
        need_scores=True,
    )

    base = make_base_row(
        row=row,
        prompt_used=result["prompt_used"],
        answer=result["answer"],
        error=result["error"],
        original_size=result["original_size"],
        processed_size=result["processed_size"],
        input_shape=result["input_shape"],
        pixel_shape=result["pixel_shape"],
        num_crops=result["num_crops"],
        run_mode=result["run_mode"],
        fallback_used=result.get("fallback_used", False),
        original_error_before_fallback=result.get("original_error_before_fallback", ""),
    )

    model_rows.append(base)

    if result["error"]:
        skipped_rows.append({
            **base,
            "pipeline": "sequence",
            "skip_reason": "generation_error",
        })
    else:
        sequence_row, scored_candidates, skip_info = analyze_numeric_sequence(result)

        for candidate in scored_candidates:
            candidate_rows.append({
                "idx": int(row["idx"]),
                "ID": row.get("ID"),
                "topic": row.get("topic"),
                "sub_topic": row.get("sub_topic"),
                "ground_truth_num": first_int(row.get("ground_truth")),
                "run_mode": result["run_mode"],
                "fallback_used": result.get("fallback_used", False),
                "original_error_before_fallback": result.get("original_error_before_fallback", ""),
                "image_resize_applied": result["run_mode"] == "resized_fallback",
                "max_image_side": MAX_IMAGE_SIDE_RETRY if result["run_mode"] == "resized_fallback" else None,
                "original_image_width": int(result["original_size"][0]),
                "original_image_height": int(result["original_size"][1]),
                "processed_image_width": int(result["processed_size"][0]),
                "processed_image_height": int(result["processed_size"][1]),
                "input_ids_shape": str(result["input_shape"]),
                "pixel_values_shape": str(result["pixel_shape"]),
                "num_image_crops": result["num_crops"],
                **candidate,
            })

        if sequence_row is not None:
            result_rows.append(sequence_row)
        else:
            skipped_rows.append({
                **base,
                "pipeline": "sequence",
                **(skip_info or {}),
            })

    processed += 1

    if processed >= next_save:
        save_outputs()
        print(f"Saved at processed={processed}")
        next_save += SAVE_EVERY

    cleanup()

LLaVA sequence numeric rows:   0%|          | 0/163 [00:00<?, ?it/s]

idx: 0 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 2 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 4 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 6 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 8 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 10 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 12 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops: 5
idx: 14 mode: original orig: (768, 512) processed: (768, 512) input_ids: (1, 2749) pixel_values: (1, 5, 3, 384, 384) crops:

In [19]:

# ============================================================
# 14. Finalize + PMI
# ============================================================

if model_rows:
    model_rows = (
        pd.DataFrame(model_rows)
        .drop_duplicates("idx", keep="last")
        .sort_values("idx")
        .to_dict("records")
    )

if candidate_rows:
    candidate_rows = (
        pd.DataFrame(candidate_rows)
        .drop_duplicates(["idx", "candidate"], keep="last")
        .sort_values(["idx", "candidate"])
        .to_dict("records")
    )

if result_rows:
    result_rows = (
        pd.DataFrame(result_rows)
        .drop_duplicates("idx", keep="last")
        .sort_values("idx")
        .to_dict("records")
    )

if skipped_rows:
    skipped_df = pd.DataFrame(skipped_rows)
    dedup_cols = ["idx", "pipeline"] if "pipeline" in skipped_df.columns else ["idx"]
    skipped_rows = (
        skipped_df
        .drop_duplicates(dedup_cols, keep="last")
        .sort_values("idx")
        .to_dict("records")
    )

# Remove skipped rows where a result exists.
if result_rows and skipped_rows:
    result_idx = {int(r["idx"]) for r in result_rows}
    skipped_rows = [
        r for r in skipped_rows
        if int(r.get("idx", -1)) not in result_idx
    ]

# Add PMI after full sequence result set is complete.
result_rows = add_pmi_columns(result_rows)

save_outputs()

pd.DataFrame(result_rows).to_csv(RESULTS_PMI_CSV, index=False)

save_parquet(model_rows, OUTPUT_DIR / "llava_original_sequence_model_outputs.parquet")
save_parquet(candidate_rows, OUTPUT_DIR / "llava_original_sequence_candidates.parquet")
save_parquet(result_rows, OUTPUT_DIR / "llava_original_sequence_results_with_pmi.parquet")
save_parquet(skipped_rows, OUTPUT_DIR / "llava_original_sequence_skipped.parquet")

ZIP_BASE = "/kaggle/working/llava_original_sequence_outputs"
ZIP_PATH = ZIP_BASE + ".zip"

if Path(ZIP_PATH).exists():
    Path(ZIP_PATH).unlink()

shutil.make_archive(ZIP_BASE, "zip", OUTPUT_DIR)

Running unconditional PMI scoring for 20 unique values:
['1', '10', '11', '12', '13', '14', '19', '2', '20', '3', '30', '32', '4', '48', '5', '50', '6', '7', '8', '9']


PMI:   0%|          | 0/20 [00:00<?, ?it/s]

'/kaggle/working/llava_original_sequence_outputs.zip'

In [20]:

# ============================================================
# 15. Summary
# ============================================================

model_df = pd.DataFrame(model_rows)
cand_df = pd.DataFrame(candidate_rows)
res_df = pd.DataFrame(result_rows)
skip_df = pd.DataFrame(skipped_rows)

print("\nDone.")
print("Model rows:", len(model_df))
print("Candidate rows:", len(cand_df))
print("Sequence result rows:", len(res_df))
print("Skipped rows:", len(skip_df))
print("ZIP:", ZIP_PATH)

if len(model_df):
    print("\nGeneration errors:")
    print(
        model_df["generation_error"]
        .fillna("")
        .replace("", "NO_ERROR")
        .value_counts(dropna=False)
        .head(30)
    )

    print("\nRun modes:")
    print(model_df["run_mode"].value_counts(dropna=False))

    print("\nFallback used:")
    print(model_df["fallback_used"].value_counts(dropna=False))

if len(res_df):
    print("\nSequence margins:")
    print(res_df[["seq_margin", "avg_margin", "char_margin", "pmi_margin"]].describe())

    print("\nPrefers gold:")
    print("avg:", res_df["prefers_gold"].mean())
    print("pmi:", res_df["prefers_gold_pmi"].mean())

    print("\nGT rank:")
    print(res_df["gt_rank_seq"].value_counts().sort_index())

    print("\nResults by run_mode:")
    print(res_df["run_mode"].value_counts(dropna=False))

if len(skip_df):
    print("\nSkip reasons:")
    print(
        skip_df["skip_reason"]
        .fillna("")
        .replace("", "NO_REASON")
        .value_counts(dropna=False)
        .head(30)
    )

print("\nSaved files:")
print(MODEL_OUTPUTS_CSV)
print(CANDIDATES_CSV)
print(RESULTS_CSV)
print(SKIPPED_CSV)
print(RESULTS_PMI_CSV)
print(ZIP_PATH)


Done.
Model rows: 163
Candidate rows: 814
Sequence result rows: 163
Skipped rows: 0
ZIP: /kaggle/working/llava_original_sequence_outputs.zip

Generation errors:
generation_error
NO_ERROR    163
Name: count, dtype: int64

Run modes:
run_mode
resized_fallback    144
original             19
Name: count, dtype: int64

Fallback used:
fallback_used
True     144
False     19
Name: count, dtype: int64

Sequence margins:
       seq_margin  avg_margin  char_margin  pmi_margin
count  163.000000  163.000000   163.000000  163.000000
mean     3.876192    3.763278     3.763278    4.051216
std      2.893715    2.938622     2.938622    3.224896
min     -4.640625   -4.640625    -4.640625   -4.021972
25%      1.390625    1.109375     1.109375    1.012207
50%      5.289063    5.281250     5.281250    5.595703
75%      6.070312    6.054688     6.054688    6.668457
max      7.277333    7.234375     7.234375    7.856445

Prefers gold:
avg: 0.8466257668711656
pmi: 0.8220858895705522

GT rank:
gt_rank_seq
1  